# Exploração da Camada Bronze — Health Insurance Marketplace

Este notebook serve como ponto de partida para a **exploração dos dados brutos** ingeridos na camada Bronze do Data Lake.

## Objetivo

- Listar todas as tabelas do database `eedb015_bronze` no AWS Glue Catalog.
- Para cada tabela, exibir as primeiras 10 linhas via **Amazon Athena** (usando `awswrangler`).
- Entender a estrutura e o conteúdo de cada tabela para guiar análises futuras nas camadas Silver e Gold.

## Como executar

1. Abra o projeto no Dev Container (`Ctrl+Shift+P` → **Dev Containers: Reopen in Container**).
2. Atualize as credenciais no arquivo `.env` e depois rode a task do VsCode **Refresh AWS Credentials**, se a sessão do Learner Lab tiver expirado.
3. Selecione o kernel **Python 3.11.14**.
4. Execute as células sequencialmente.

> **Atenção:** este notebook usa `awswrangler` para consultar o Athena — diferente do `landing_to_bronze.ipynb` que usa PySpark/Glue. Não é necessário inicializar SparkContext aqui.

---
## 1. Configuração e Imports

In [14]:
import awswrangler as wr
import boto3
import pandas as pd
from IPython.display import Markdown, display

# Exibe todas as colunas e evita truncamento
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_rows", None)
pd.set_option("display.width", 300)

In [2]:
# Descobre o Account ID dinamicamente — funciona para qualquer membro do grupo
account_id = boto3.client("sts").get_caller_identity()["Account"]

BRONZE_DATABASE   = "eedb015_bronze"
S3_ATHENA_OUTPUT  = f"s3://{account_id}-eedb015-athena-results/"

print(f"Account ID      : {account_id}")
print(f"Bronze Database : {BRONZE_DATABASE}")
print(f"Athena Output   : {S3_ATHENA_OUTPUT}")

Account ID      : 637423524537
Bronze Database : eedb015_bronze
Athena Output   : s3://637423524537-eedb015-athena-results/


---
## 2. Descoberta de Tabelas

In [3]:
# Lista todas as tabelas do database Bronze no Glue Catalog
tables_df = wr.catalog.tables(database=BRONZE_DATABASE)

print(f"Total de tabelas no database '{BRONZE_DATABASE}': {len(tables_df)}")
display(tables_df[["Table", "TableType"]].sort_values("Table").reset_index(drop=True))

Total de tabelas no database 'eedb015_bronze': 42


,Table,TableType
0,benefits_cost_sharing,EXTERNAL_TABLE
1,business_rules,EXTERNAL_TABLE
2,crosswalk2015,EXTERNAL_TABLE
3,crosswalk2016,EXTERNAL_TABLE
4,network,EXTERNAL_TABLE
5,plan_attributes,EXTERNAL_TABLE
6,rate,EXTERNAL_TABLE
7,raw_2014_benefits_cost_sharing_puf,EXTERNAL_TABLE
8,raw_2014_business_rules_puf,EXTERNAL_TABLE
9,raw_2014_network_puf,EXTERNAL_TABLE


In [4]:
# Funções auxiliares usadas nas seções seguintes

all_table_names: list[str] = sorted(tables_df["Table"].tolist())


def tables_by_pattern(*patterns: str) -> list[str]:
    """Retorna tabelas cujo nome contenha qualquer um dos padrões fornecidos."""
    result = []
    for t in all_table_names:
        if any(p in t for p in patterns):
            result.append(t)
    return result


def preview_table(table_name: str, limit: int = 10) -> pd.DataFrame:
    """Retorna as primeiras `limit` linhas de uma tabela via Athena."""
    sql = f'SELECT * FROM "{BRONZE_DATABASE}"."{table_name}" LIMIT {limit}'
    try:
        df = wr.athena.read_sql_query(
            sql=sql,
            database=BRONZE_DATABASE,
            s3_output=S3_ATHENA_OUTPUT,
            ctas_approach=False,
        )
        return df
    except Exception as exc:
        print(f"[ERRO] Não foi possível consultar '{table_name}': {exc}")
        return pd.DataFrame()


def show_family(patterns: list[str]) -> None:
    """Exibe preview de todas as tabelas que casam com os padrões informados, excluindo as que contêm 'raw' no nome."""
    matched = [t for t in tables_by_pattern(*patterns) if "raw" not in t]
    if not matched:
        print("Nenhuma tabela encontrada para este padrão.")
        return
    for table in matched:
        display(Markdown(f"### `{table}`"))
        df = preview_table(table)
        if not df.empty:
            display(df)
            print(f"   ↳ {len(df.columns)} colunas | {len(df)} linhas exibidas")
        print()

In [11]:
PROFILE_SAMPLE = 50_000

# Valores sentinela comuns no dataset — tratados como nulos conceituais
_SENTINELS = {
    "", "not applicable", "not available", "n/a", "na", "none", "null",
    "not offered", "no charge", "no charge after deductible",
    "covered", "uncovered", "varies", "see contract",
}


def _is_sentinel(s: str) -> bool:
    return isinstance(s, str) and s.strip().lower() in _SENTINELS


def _try_numeric(series: pd.Series) -> pd.Series | None:
    """Tenta converter coluna para numérico, removendo prefixos como $, %, vírgulas."""
    cleaned = series.astype(str).str.replace(r'[$%,]', '', regex=True).str.strip()
    converted = pd.to_numeric(cleaned, errors='coerce')
    # Aceita como numérica se >= 60 % dos não-nulos converteram
    valid_ratio = converted.notna().sum() / max(series.notna().sum(), 1)
    return converted if valid_ratio >= 0.6 else None


def _try_date(series: pd.Series) -> pd.Series | None:
    """Tenta converter coluna para data."""
    converted = pd.to_datetime(series.astype(str).str.strip(), errors='coerce', format='mixed')
    valid_ratio = converted.notna().sum() / max(series.notna().sum(), 1)
    return converted if valid_ratio >= 0.8 else None


def profile_table(table_name: str) -> None:
    """Perfil detalhado de uma tabela Bronze:
    - Contagem total de linhas (Athena COUNT)
    - Amostra de até 50.000 linhas para análise local
    - Por coluna: nulos, sentinelas, cardinalidade, tipo inferido,
      estatísticas numéricas, top valores frequentes
    - Linhas duplicadas exatas
    """
    display(Markdown(f'## Perfil: `{table_name}`'))

    # --- 1. Contagem total ---
    try:
        count_df = wr.athena.read_sql_query(
            sql=f'SELECT COUNT(*) AS total FROM "{BRONZE_DATABASE}"."{table_name}"',
            database=BRONZE_DATABASE,
            s3_output=S3_ATHENA_OUTPUT,
            ctas_approach=False,
        )
        total_rows = int(count_df['total'].iloc[0])
    except Exception as exc:
        print(f'[ERRO] COUNT falhou: {exc}')
        return

    print(f'Total de linhas na tabela: {total_rows:,}')

    # --- 2. Amostra ---
    try:
        df = wr.athena.read_sql_query(
            sql=f'SELECT * FROM "{BRONZE_DATABASE}"."{table_name}" TABLESAMPLE BERNOULLI(25) ORDER BY random() LIMIT {PROFILE_SAMPLE}',
            database=BRONZE_DATABASE,
            s3_output=S3_ATHENA_OUTPUT,
            ctas_approach=False,
        )
    except Exception as exc:
        print(f'[ERRO] Amostra falhou: {exc}')
        return

    sample_size = len(df)
    n_cols = len(df.columns)
    print(f'Amostra carregada: {sample_size:,} linhas × {n_cols} colunas')

    # --- 3. Duplicatas exatas ---
    n_dupes = df.duplicated().sum()
    dupe_pct = n_dupes / sample_size * 100 if sample_size else 0
    dupe_flag = ' ⚠️' if n_dupes > 0 else ' ✅'
    print(f'Linhas duplicadas na amostra: {n_dupes:,} ({dupe_pct:.1f}%){dupe_flag}')
    print()

    # --- 4. Perfil por coluna ---
    rows = []
    suggestions = []

    for col in df.columns:
        s = df[col]

        # Nulos reais (None/NaN)
        null_count = s.isna().sum()
        null_pct = null_count / sample_size * 100

        # Sentinelas (nulos semânticos)
        sentinel_count = s.dropna().apply(_is_sentinel).astype(bool).sum()
        sentinel_pct = sentinel_count / sample_size * 100

        # Valores não-nulos para análise
        s_valid = s.dropna()
        s_valid = s_valid[~s_valid.apply(_is_sentinel)]

        unique_count = s.nunique(dropna=True)
        cardinality = 'alta' if unique_count > 100 else ('média' if unique_count > 10 else 'baixa')

        # Inferência de tipo
        inferred_type = 'string'
        extra_stats = ''
        top_values = ''
        treatment = []

        numeric_series = _try_numeric(s_valid) if len(s_valid) > 0 else None
        date_series = None

        if numeric_series is not None and numeric_series.notna().sum() > 0:
            inferred_type = 'numeric'
            mn, mx, mean, std = (
                numeric_series.min(), numeric_series.max(),
                numeric_series.mean(), numeric_series.std()
            )
            extra_stats = f'min={mn:.2f} max={mx:.2f} mean={mean:.2f} std={std:.2f}'
            treatment.append('cast → double/decimal')
            if mn < 0:
                treatment.append('verificar negativos')
                suggestions.append(f'`{col}`: valores negativos (min={mn:.2f})')
            if std > 0 and (mx - mean) > 4 * std:
                treatment.append('possíveis outliers (>4σ)')
                suggestions.append(f'`{col}`: outliers detectados (max={mx:.2f}, mean={mean:.2f})')
        else:
            date_series = _try_date(s_valid) if len(s_valid) > 0 else None
            if date_series is not None:
                inferred_type = 'date'
                mn, mx = date_series.min(), date_series.max()
                extra_stats = f'de {mn.date()} até {mx.date()}'
                treatment.append('cast → date/timestamp')
            else:
                # string — top valores
                top = s_valid.astype(str).value_counts().head(5)
                top_values = ' | '.join([f'{v}({c})' for v, c in top.items()])
                if unique_count <= 30:
                    treatment.append('candidata a enum/categórica')

        if null_pct > 50:
            treatment.append(f'alta nulidade ({null_pct:.0f}%)')
            suggestions.append(f'`{col}`: {null_pct:.0f}% nulos — verificar se é campo opcional')
        if sentinel_count > 0:
            treatment.append(f'sentinelas → null ({sentinel_count:,})')

        rows.append({
            'coluna': col,
            'tipo_inferido': inferred_type,
            'nulos': f'{null_count:,} ({null_pct:.1f}%)',
            'sentinelas': f'{sentinel_count:,} ({sentinel_pct:.1f}%)' if sentinel_count > 0 else '—',
            'únicos': f'{unique_count:,} [{cardinality}]',
            'stats / top valores': extra_stats or top_values or '—',
            'tratamentos sugeridos': ' | '.join(treatment) if treatment else '—',
        })

    profile_df = pd.DataFrame(rows).set_index('coluna')
    display(profile_df)

    # --- 5. Resumo de ações ---
    if suggestions:
        display(Markdown('### ⚠️ Pontos de Atenção'))
        for s in suggestions:
            print(f'  • {s}')

    print()


def profile_family(patterns: list[str]) -> None:
    """Roda profile_table para todas as tabelas da família, excluindo as com 'raw'."""
    matched = [t for t in tables_by_pattern(*patterns) if 'raw' not in t]
    if not matched:
        print('Nenhuma tabela encontrada para este padrão.')
        return
    for table in matched:
        profile_table(table)


---
## 3. Tabela: Rate — Tarifas dos Planos

**O que é:** Contém os **prêmios mensais** cobrados por cada plano de saúde, segmentados por faixa etária, perfil de tabaco e área geográfica de tarifação. É a **maior tabela do dataset** — aproximadamente 50 milhões de linhas no período 2014–2016, pois cada plano gera ~50–80 linhas (uma por combinação de idade × área geográfica × perfil de tabaco).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano (2014, 2015 ou 2016) |
| `StateCode` | string | Estado norte-americano (2 letras, ex: `CA`, `TX`) |
| `IssuerId` | string | Código da seguradora emissora |
| `SourceName` | string | Fonte/exchange (ex: `HIOS`, `SBM`) |
| `VersionNum` | integer | Número de versão da submissão |
| `ImportDate` | date | Data de importação do arquivo |
| `IssuerId2` | string | Identificador alternativo da seguradora |
| `FederalTIN` | string | Federal Tax Identification Number |
| `RateEffectiveDate` | date | Início de vigência da tarifa |
| `RateExpirationDate` | date | Término de vigência da tarifa |
| `PlanId` | string | ID do plano no formato HIOS — chave de ligação com `PlanAttributes` |
| `RatingAreaId` | string | ID da área de tarifação (sub-região geográfica) |
| `Tobacco` | string | Perfil de tabaco: `Tobacco User` / `No Preference` |
| `Age` | string | Faixa etária: `0-20`, `21`, `27`, `30`, ..., `64`, `Family Option` |
| `IndividualRate` | decimal | Prêmio mensal individual em USD |
| `IndividualTobaccoRate` | decimal | Prêmio para fumante individual |
| `Couple` | decimal | Prêmio para casal |
| `PrimarySubscriberAndOneDependent` | decimal | Titular + 1 dependente |
| `PrimarySubscriberAndTwoDependents` | decimal | Titular + 2 dependentes |
| `PrimarySubscriberAndThreeOrMoreDependents` | decimal | Titular + 3+ dependentes |
| `CoupleAndOneDependent` | decimal | Casal + 1 dependente |
| `CoupleAndTwoDependents` | decimal | Casal + 2 dependentes |
| `CoupleAndThreeOrMoreDependents` | decimal | Casal + 3+ dependentes |

**Relações:** `PlanId` → `PlanAttributes.PlanId`; `IssuerId` + `StateCode` → `ServiceArea`

**Insights dos notebooks Kaggle:**
- Prêmios para 64 anos chegam a ser **3× o prêmio de 21 anos** no mesmo plano (limite máximo permitido pelo ACA).
- A variável `Tobacco` adiciona surcharge de ~50% sobre o prêmio base (teto definido em lei).
- Estados com maior competição (10+ seguradoras) apresentam prêmios médios menores — correlação negativa especialmente no nível `Silver`.

**Relevância para o projeto:** Responde diretamente às questões 2 (competição × preço), 3 (precificação) e 4 (rede × preço). É a tabela de fato principal para análises de valor.

> ⚠️ **Nota de escala:** Por ser a maior tabela (~50M linhas), consultas no Athena sem filtro de `StateCode` ou `BusinessYear` são lentas e custosas — sempre filtre antes de agregar.

In [6]:
show_family(["rate"])

### `rate`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,0-20,29.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
1,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,21,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
2,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,22,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
3,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 3,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
4,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 1,No Preference,Family Option,32.45,<NA>,64.9,94.5,94.5,94.5,126.95,126.95,126.95,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
5,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,23,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
6,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,24,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
7,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 2,No Preference,Family Option,32.45,<NA>,64.9,94.5,94.5,94.5,126.95,126.95,126.95,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
8,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 1,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
9,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 2,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00


   ↳ 26 colunas | 10 linhas exibidas



In [8]:
profile_family(["rate"])      # toda a família

## Perfil: `rate`

Total de linhas na tabela: 12,694,445
Amostra carregada: 50,000 linhas × 26 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.03 std=0.80,cast → double/decimal
statecode,string,0 (0.0%),—,39 [média],FL(6775) | SC(6333) | MI(3926) | WI(3925) | OH(3521),—
issuerid,numeric,0 (0.0%),—,824 [alta],min=10046.00 max=99969.00 mean=52552.78 std=26289.66,cast → double/decimal
sourcename,string,0 (0.0%),—,3 [baixa],HIOS(34389) | SERFF(15070) | OPM(541),candidata a enum/categórica
versionnum,numeric,0 (0.0%),—,23 [média],min=1.00 max=24.00 mean=6.88 std=3.87,cast → double/decimal | possíveis outliers (>4σ)
importdate,date,0 (0.0%),—,262 [alta],de 2013-06-04 até 2015-11-21,cast → date/timestamp
issuerid2,numeric,0 (0.0%),—,824 [alta],min=10046.00 max=99969.00 mean=52552.78 std=26289.66,cast → double/decimal
federaltin,string,0 (0.0%),—,329 [alta],57-0768835(2841) | 47-0397286(2412) | 36-1236610(2067) | 13-5581829(1676) | 95-6042390(1368),—
rateeffectivedate,date,0 (0.0%),—,12 [média],de 2014-01-01 até 2016-10-01,cast → date/timestamp


### ⚠️ Pontos de Atenção

  • `versionnum`: outliers detectados (max=24.00, mean=6.88)
  • `individualrate`: outliers detectados (max=999999.00, mean=3777.83)
  • `individualtobaccorate`: outliers detectados (max=5985.29, mean=546.93)
  • `individualtobaccorate`: 61% nulos — verificar se é campo opcional
  • `couple`: 100% nulos — verificar se é campo opcional
  • `primarysubscriberandonedependent`: 100% nulos — verificar se é campo opcional
  • `primarysubscriberandtwodependents`: 100% nulos — verificar se é campo opcional
  • `primarysubscriberandthreeormoredependents`: 100% nulos — verificar se é campo opcional
  • `coupleandonedependent`: 100% nulos — verificar se é campo opcional
  • `coupleandtwodependents`: 100% nulos — verificar se é campo opcional
  • `coupleandthreeormoredependents`: 100% nulos — verificar se é campo opcional
  • `rownumber`: outliers detectados (max=63406.00, mean=6413.86)



---
## 4. Tabela: PlanAttributes — Atributos dos Planos

**O que é:** Metadados descritivos de cada plano: nível metálico, tipo de rede, deductibles, out-of-pocket máximos, cobertura de saúde mental, dental, visão, etc. Atua como o **catálogo mestre dos planos** — a tabela dimensional central do dataset (~11.000 planos por ano).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `PlanId` | string | Identificador único HIOS do plano (chave primária) |
| `PlanMarketingName` | string | Nome comercial do plano (ex: "Blue Shield Silver 70") |
| `HIOSProductId` | string | ID do produto no sistema HIOS |
| `NetworkId` | string | Identificador da rede de prestadores |
| `ServiceAreaId` | string | ID da área de serviço — chave para `ServiceArea` |
| `FormularyId` | string | ID do formulário farmacêutico |
| `IsNewPlan` | string | Plano novo naquele ano (`Yes`/`No`) |
| `PlanType` | string | Tipo de rede: `HMO`, `PPO`, `EPO`, `POS`, `PFFS`, `HDHP` |
| `MetalLevel` | string | Nível metálico: `Bronze`, `Silver`, `Gold`, `Platinum`, `Catastrophic` |
| `QHPorSADPOnly` | string | Qualified Health Plan ou Standalone Dental Plan |
| `ChildOnlyOffering` | string | Plano disponível para crianças apenas |
| `WellnessProgramOffered` | string | Possui programa de bem-estar (`Yes`/`No`) |
| `DiseaseManagementProgramsOffered` | string | Programas de gestão de doenças crônicas |
| `NationalNetwork` | string | Cobertura nacional (`Yes`/`No`) |
| `EHBPercentTotalPremium` | decimal | % do prêmio referente a Essential Health Benefits |
| `CSRVariationType` | string | Tipo de variação de Cost-Sharing Reduction (para segurados de baixa renda) |
| `MEHBInnTier1IndividualMOOP` | decimal | Maximum Out-of-Pocket individual — médico in-network Tier 1 |
| `MEHBInnTier1FamilyMOOP` | decimal | MOOP familiar — médico in-network Tier 1 |
| `MEHBOutOfNetIndividualMOOP` | decimal | MOOP individual fora da rede |
| `MEHBInnTier1IndividualDeductible` | decimal | Deductible individual médico in-network |
| `MEHBInnTier1FamilyDeductible` | decimal | Deductible familiar médico in-network |
| `SBCHavingaBabyDeductible` | decimal | Exemplo SBC — custo de ter um bebê (deductible) |
| `SBCHavingDiabetesCopayment` | decimal | Exemplo SBC — copay anual para diabetes |
| `SBCHavingSimpleFractureDeductible` | decimal | Exemplo SBC — deductible para fratura simples |

**Nota sobre `PlanId` (formato HIOS):** `XXXXX-XX-XXXXXXXX` — `IssuerId (5 dígitos) + ProductId (2) + PlanId (2) + VariantId (2)`. Variantes com sufixo `73` representam o mesmo plano com deductibles reduzidos para elegíveis ao subsídio CSR — devem ser agrupadas com o plano base nos joins.

**Relações:** Tabela dimensional central. Liga-se com `Rate` (via `PlanId`), `BenefitsCostSharing` (via `PlanId`), `ServiceArea` (via `ServiceAreaId`) e `Network` (via `NetworkId`).

**Insights dos notebooks Kaggle:**
- `MetalLevel = Silver` é o mais comum (~40% dos planos) por ser o nível de referência para os subsídios federais (Cost-Sharing Reductions).
- HMOs dominam mercados urbanos densos; PPOs predominam em estados com menor densidade de prestadores.
- O número de `IssuerId` distintos por `StateCode` é o principal proxy de competição para a Questão 2.

**Relevância para o projeto:** Chave de segmentação em todas as questões — `MetalLevel` e `PlanType` são as variáveis de classificação mais usadas nas análises.

In [12]:
show_family(["plan_attributes"])

### `plan_attributes`

,avcalculatoroutputnumber,beginprimarycarecostsharingafternumberofvisits,beginprimarycaredeductiblecoinsuranceafternumberofcopays,benefitpackageid,businessyear,csrvariationtype,childonlyoffering,childonlyplanid,compositeratingoffered,dehbcombinnoonfamilymoop,dehbcombinnoonfamilypergroupmoop,dehbcombinnoonfamilyperpersonmoop,dehbcombinnoonindividualmoop,dehbdedcombinnoonfamily,dehbdedcombinnoonfamilypergroup,dehbdedcombinnoonfamilyperperson,dehbdedcombinnoonindividual,dehbdedinntier1coinsurance,dehbdedinntier1family,dehbdedinntier1familypergroup,dehbdedinntier1familyperperson,dehbdedinntier1individual,dehbdedinntier2coinsurance,dehbdedinntier2family,dehbdedinntier2familypergroup,dehbdedinntier2familyperperson,dehbdedinntier2individual,dehbdedoutofnetfamily,dehbdedoutofnetfamilypergroup,dehbdedoutofnetfamilyperperson,dehbdedoutofnetindividual,dehbinntier1familymoop,dehbinntier1familypergroupmoop,dehbinntier1familyperpersonmoop,dehbinntier1individualmoop,dehbinntier2familymoop,dehbinntier2familypergroupmoop,dehbinntier2familyperpersonmoop,dehbinntier2individualmoop,dehboutofnetfamilymoop,dehboutofnetfamilypergroupmoop,dehboutofnetfamilyperpersonmoop,dehboutofnetindividualmoop,dentalonlyplan,diseasemanagementprogramsoffered,ehbpediatricdentalapportionmentquantity,ehbpercentpremiums4,ehbpercenttotalpremium,firsttierutilization,formularyid,formularyurl,hiosproductid,hpid,hsaorhraemployercontribution,hsaorhraemployercontributionamount,importdate,indianplanvariationestimatedadvancedpaymentamountperenrollee,inpatientcopaymentmaximumdays,isguaranteedrate,ishsaeligible,isnewplan,isnoticerequiredforpregnancy,isreferralrequiredforspecialist,issueractuarialvalue,issuerid,issuerid2,mehbcombinnoonfamilymoop,mehbcombinnoonfamilypergroupmoop,mehbcombinnoonfamilyperpersonmoop,mehbcombinnoonindividualmoop,mehbdedcombinnoonfamily,mehbdedcombinnoonfamilypergroup,mehbdedcombinnoonfamilyperperson,mehbdedcombinnoonindividual,mehbdedinntier1coinsurance,mehbdedinntier1family,mehbdedinntier1familypergroup,mehbdedinntier1familyperperson,mehbdedinntier1individual,mehbdedinntier2coinsurance,mehbdedinntier2family,mehbdedinntier2familypergroup,mehbdedinntier2familyperperson,mehbdedinntier2individual,mehbdedoutofnetfamily,mehbdedoutofnetfamilypergroup,mehbdedoutofnetfamilyperperson,mehbdedoutofnetindividual,mehbinntier1familymoop,mehbinntier1familypergroupmoop,mehbinntier1familyperpersonmoop,mehbinntier1individualmoop,mehbinntier2familymoop,mehbinntier2familypergroupmoop,mehbinntier2familyperpersonmoop,mehbinntier2individualmoop,mehboutofnetfamilymoop,mehboutofnetfamilypergroupmoop,mehboutofnetfamilyperpersonmoop,mehboutofnetindividualmoop,marketcoverage,medicaldrugdeductiblesintegrated,medicaldrugmaximumoutofpocketintegrated,metallevel,multipleinnetworktiers,nationalnetwork,networkid,outofcountrycoverage,outofcountrycoveragedescription,outofserviceareacoverage,outofserviceareacoveragedescription,planbrochure,planeffictivedate,planexpirationdate,planid,planlevelexclusions,planmarketingname,plantype,qhpnonqhptypeid,rownumber,sbchavingdiabetescoinsurance,sbchavingdiabetescopayment,sbchavingdiabetesdeductible,sbchavingdiabeteslimit,sbchavingababycoinsurance,sbchavingababycopayment,sbchavingababydeductible,sbchavingababylimit,secondtierutilization,serviceareaid,sourcename,specialistrequiringreferral,specialtydrugmaximumcoinsurance,standardcomponentid,statecode,statecode2,tehbcombinnoonfamilymoop,tehbcombinnoonfamilypergroupmoop,tehbcombinnoonfamilyperpersonmoop,tehbcombinnoonindividualmoop,tehbdedcombinnoonfamily,tehbdedcombinnoonfamilypergroup,tehbdedcombinnoonfamilyperperson,tehbdedcombinnoonindividual,tehbdedinntier1coinsurance,tehbdedinntier1family,tehbdedinntier1familypergroup,tehbdedinntier1familyperperson,tehbdedinntier1individual,tehbdedinntier2coinsurance,tehbdedinntier2family,tehbdedinntier2familypergroup,tehbdedinntier2familyperperson,tehbdedinntier2individual,tehbdedoutofnetfamily,tehbdedoutofnetfamilypergroup,tehbdedoutofnetfamilyperperson,tehbdedouto

   ↳ 178 colunas | 10 linhas exibidas



In [15]:
profile_family(["plan_attributes"])

## Perfil: `plan_attributes`

Total de linhas na tabela: 77,353
Amostra carregada: 19,205 linhas × 178 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
avcalculatoroutputnumber,numeric,"5,586 (29.1%)",—,"4,414 [alta]",min=0.00 max=100.00 mean=0.84 std=2.83,cast → double/decimal | possíveis outliers (>4σ)
beginprimarycarecostsharingafternumberofvisits,numeric,0 (0.0%),—,7 [baixa],min=0.00 max=6.00 mean=0.15 std=0.66,cast → double/decimal | possíveis outliers (>4σ)
beginprimarycaredeductiblecoinsuranceafternumberofcopays,numeric,0 (0.0%),—,8 [baixa],min=0.00 max=10.00 mean=0.30 std=1.22,cast → double/decimal | possíveis outliers (>4σ)
benefitpackageid,numeric,0 (0.0%),—,64 [média],min=1.00 max=84.00 mean=4.92 std=6.59,cast → double/decimal | possíveis outliers (>4σ)
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.11 std=0.76,cast → double/decimal
csrvariationtype,string,0 (0.0%),—,21 [média],Zero Cost Sharing Plan Variation(2705) | Limited Cost Sharing Plan Variation(2670) | Standard Si...,candidata a enum/categórica
childonlyoffering,string,0 (0.0%),—,4 [baixa],Allows Adult and Child-Only(18202) | Allows Child-Only(870) | Allows Adult-Only(130) | Allows Ad...,candidata a enum/categórica
childonlyplanid,string,"19,088 (99.4%)",—,52 [média],29698MI0540006(7) | 99969OH0040019(6) | 99969OH0040017(5) | 23340OH0010012(4) | 29241MI0270066(4),alta nulidade (99%)
compositeratingoffered,string,"12,452 (64.8%)",—,2 [baixa],No(6521) | Yes(232),candidata a enum/categórica | alta nulidade (65%)


### ⚠️ Pontos de Atenção

  • `avcalculatoroutputnumber`: outliers detectados (max=100.00, mean=0.84)
  • `beginprimarycarecostsharingafternumberofvisits`: outliers detectados (max=6.00, mean=0.15)
  • `beginprimarycaredeductiblecoinsuranceafternumberofcopays`: outliers detectados (max=10.00, mean=0.30)
  • `benefitpackageid`: outliers detectados (max=84.00, mean=4.92)
  • `childonlyplanid`: 99% nulos — verificar se é campo opcional
  • `compositeratingoffered`: 65% nulos — verificar se é campo opcional
  • `dehbcombinnoonfamilymoop`: 99% nulos — verificar se é campo opcional
  • `dehbcombinnoonfamilypergroupmoop`: 100% nulos — verificar se é campo opcional
  • `dehbcombinnoonfamilyperpersonmoop`: 100% nulos — verificar se é campo opcional
  • `dehbcombinnoonindividualmoop`: 98% nulos — verificar se é campo opcional
  • `dehbdedcombinnoonfamily`: outliers detectados (max=1500.00, mean=91.90)
  • `dehbdedcombinnoonfamily`: 76% nulos — verificar se é campo opcional
  • `dehbdedcombinnoonfamilypergroup`: 88% nulos

---
## 5. Tabela: BenefitsCostSharing — Benefícios e Compartilhamento de Custos

**O que é:** Detalha os **benefícios cobertos** por cada plano e as regras de compartilhamento de custo que o segurado paga ao utilizar cada serviço. Cada linha representa um benefício específico dentro de um plano — é a tabela mais granular do dataset em termos de coberturas.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano |
| `StateCode` | string | Estado |
| `IssuerId` | string | ID da seguradora |
| `PlanId` | string | Chave de ligação com `PlanAttributes` |
| `BenefitName` | string | Nome do benefício (ver lista abaixo) |
| `IsCovered` | string | Se o benefício é coberto: `Covered` / `Not Covered` |
| `QuantLimitOnSvc` | string | Há limite quantitativo de serviços? (`Yes`/`No`) |
| `LimitQty` | decimal | Quantidade máxima coberta (ex: 30 visitas/ano) |
| `LimitUnit` | string | Unidade do limite: `Per Year`, `Per Visit`, `Days` |
| `Exclusions` | string | Descrição de exclusões específicas |
| `Explanation` | string | Explicação adicional sobre o benefício |
| `EHBInd` | string | É um Essential Health Benefit obrigatório pelo ACA? |
| `IsSubjToDedBefore` | string | O cost-sharing está sujeito ao deductible antes de ser aplicado? |
| `CoinsInnTier1` | string | Coinsurance (%) — prestador in-network Tier 1 |
| `CoinsInnTier2` | string | Coinsurance (%) — prestador in-network Tier 2 |
| `CoinsOutofNet` | string | Coinsurance (%) — prestador fora da rede |
| `CopayInnTier1` | string | Copay ($) — prestador in-network Tier 1 |
| `CopayInnTier2` | string | Copay ($) — prestador in-network Tier 2 |
| `CopayOutofNet` | string | Copay ($) — prestador fora da rede |

**Benefícios de maior relevância para o projeto:**

| BenefitName | Relevância |
|---|---|
| `Chemotherapy` | Questão 1 — evolução do custo para pacientes oncológicos |
| `Radiation Therapy` | Questão 1 — custo de radioterapia ao longo dos anos |
| `Infusion Therapy` | Questão 1 — terapia recorrente para doenças crônicas |
| `Specialist Visit` | Questão 3 — benefício como variável de precificação |
| `Primary Care Visit to Treat an Injury or Illness` | Questão 3 |
| `Emergency Room Services` | Questão 3 |
| `Mental/Behavioral Health Outpatient Services` | Questão 3 |

**Relações:** `PlanId` → `PlanAttributes.PlanId` (e transitivamente → `Rate.PlanId`).

**Insights dos notebooks Kaggle (Ryan Burge — Cancer and the ACA):**
- Entre 2014 e 2016, a proporção de planos com **coinsurance zero** para quimioterapia aumentou — melhora geral na cobertura oncológica.
- Planos **Gold** e **Platinum** tendem a ter copay fixo para quimioterapia, protegendo o paciente de custos variáveis em tratamentos longos.
- Planos **Bronze** frequentemente aplicam coinsurance de 40–50% após o deductible, expondo pacientes crônicos a custos catastróficos.
- `CopayInnTier2` e `CoinsInnTier2` têm alta taxa de nulos — muitos planos usam apenas Tier 1.

**Atenção ao parsing:** Os campos de copay/coinsurance chegam como strings mistas: `"$30"`, `"$0"`, `"No Charge"`, `"Not Applicable"`. O ETL da camada Silver precisa normalizar esses valores para decimais.

**Relevância para o projeto:** **Principal tabela para a Questão 1** (evolução Copay × Coinsurance em oncologia) e fundamental para a Questão 3 (benefícios como variável de precificação).

In [16]:
show_family(["benefits_cost_sharing", "benefits"])

### `benefits_cost_sharing`

,benefitname,businessyear,coinsinntier1,coinsinntier2,coinsoutofnet,copayinntier1,copayinntier2,copayoutofnet,ehbvarreason,exclusions,explanation,importdate,iscovered,isehb,isexclfrominnmoop,isexclfromoonmoop,isstatemandate,issubjtodedtier1,issubjtodedtier2,issuerid,issuerid2,limitqty,limitunit,minimumstay,planid,quantlimitonsvc,rownumber,sourcename,standardcomponentid,statecode,statecode2,versionnum,landinzone_path,ingestion_datetime
0,Routine Dental Services (Adult),2014,20%,<NA>,20%,No Charge,<NA>,No Charge,Above EHB,<NA>,Combined annual benefit maximum of $1000 per year for adults. See policy for additional limitations,2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,No,No,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,68,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
1,Dental Check-Up for Children,2014,20%,<NA>,20%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,No,No,21989,21989,1.0,Visit(s) per 6 Months,<NA>,21989AK0010001-00,Yes,104,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
2,Basic Dental Care - Child,2014,40%,<NA>,40%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,110,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
3,Orthodontia - Child,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Additional EHB Benefit,<NA>,"24 month waiting period, See policy for additional limitations",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,111,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
4,Major Dental Care - Child,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,112,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
5,Basic Dental Care - Adult,2014,40%,<NA>,40%,No Charge,<NA>,No Charge,Above EHB,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 6 month waiting period, See policy...",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,113,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
6,Orthodontia - Adult,2014,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2014-03-19 07:06:49,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,114,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
7,Major Dental Care - Adult,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Above EHB,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 12 month waiting period, See polic...",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,115,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
8,Accidental Dental,2014,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2014-03-19 07:06:49,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,118,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
9,Routine Dental Servi

   ↳ 34 colunas | 10 linhas exibidas



In [17]:
profile_family(["benefits"])

## Perfil: `benefits_cost_sharing`

Total de linhas na tabela: 5,048,468
Amostra carregada: 50,000 linhas × 34 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
benefitname,string,0 (0.0%),—,373 [alta],Dental Check-Up for Children(792) | Routine Dental Services (Adult)(787) | Basic Dental Care - A...,—
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.12 std=0.76,cast → double/decimal
coinsinntier1,string,"11,103 (22.2%)","12,763 (25.5%)",51 [média],20% Coinsurance after deductible(6758) | 0%(6158) | 30% Coinsurance after deductible(3327) | 0% ...,"sentinelas → null (12,763)"
coinsinntier2,string,"45,215 (90.4%)","2,763 (5.5%)",29 [média],0%(777) | 40% Coinsurance after deductible(281) | 50% Coinsurance after deductible(237) | 30% Co...,"candidata a enum/categórica | alta nulidade (90%) | sentinelas → null (2,763)"
coinsoutofnet,numeric,"11,103 (22.2%)","2,025 (4.0%)",42 [média],min=0.00 max=100.00 mean=83.12 std=36.59,"cast → double/decimal | sentinelas → null (2,025)"
copayinntier1,numeric,"11,103 (22.2%)","24,959 (49.9%)",239 [alta],min=0.00 max=5000.00 mean=32.26 std=143.26,"cast → double/decimal | possíveis outliers (>4σ) | sentinelas → null (24,959)"
copayinntier2,numeric,"45,215 (90.4%)","3,233 (6.5%)",118 [alta],min=0.00 max=5000.00 mean=51.08 std=311.94,"cast → double/decimal | possíveis outliers (>4σ) | alta nulidade (90%) | sentinelas → null (3,233)"
copayoutofnet,numeric,"11,103 (22.2%)","25,969 (51.9%)",110 [alta],min=0.00 max=1000.00 mean=5.68 std=39.90,"cast → double/decimal | possíveis outliers (>4σ) | sentinelas → null (25,969)"
ehbvarreason,string,"29,877 (59.8%)",—,7 [baixa],Substantially Equal(7973) | Additional EHB Benefit(6811) | Above EHB(2499) | Dental Only Plan Av...,candidata a enum/categórica | alta nulidade (60%)


### ⚠️ Pontos de Atenção

  • `coinsinntier2`: 90% nulos — verificar se é campo opcional
  • `copayinntier1`: outliers detectados (max=5000.00, mean=32.26)
  • `copayinntier2`: outliers detectados (max=5000.00, mean=51.08)
  • `copayinntier2`: 90% nulos — verificar se é campo opcional
  • `copayoutofnet`: outliers detectados (max=1000.00, mean=5.68)
  • `ehbvarreason`: 60% nulos — verificar se é campo opcional
  • `exclusions`: 90% nulos — verificar se é campo opcional
  • `explanation`: 81% nulos — verificar se é campo opcional
  • `isstatemandate`: 84% nulos — verificar se é campo opcional
  • `limitqty`: outliers detectados (max=89942.00, mean=212.35)
  • `limitqty`: 86% nulos — verificar se é campo opcional
  • `limitunit`: 86% nulos — verificar se é campo opcional
  • `minimumstay`: outliers detectados (max=38345.00, mean=1090.28)
  • `minimumstay`: 100% nulos — verificar se é campo opcional
  • `quantlimitonsvc`: 64% nulos — verificar se é campo opcional
  • `versionnum`: outliers detectados (max=24.00, m

---
## 6. Tabela: ServiceArea — Áreas de Serviço

**O que é:** Define as **regiões geográficas** onde cada seguradora está autorizada a oferecer planos. A granularidade pode ser estado inteiro, condados específicos ou até sub-divisões de condados (partial county / ZIP codes). É o arquivo que permite mapear "desertos de cobertura".

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `SourceName` | string | Fonte/exchange |
| `VersionNum` | integer | Versão da submissão |
| `ImportDate` | date | Data de importação |
| `IssuerId2` | string | ID alternativo |
| `FederalTIN` | string | TIN federal |
| `ServiceAreaId` | string | Identificador único da área de serviço (chave para `PlanAttributes`) |
| `ServiceAreaName` | string | Nome descritivo da área (ex: "California Statewide") |
| `CoverEntireState` | string | Cobre o estado inteiro? (`Yes`/`No`) |
| `County` | string | Nome do condado coberto (quando não cobre o estado todo) |
| `CountyCode` | string | Código FIPS do condado |
| `PartialCounty` | string | Cobertura parcial de condado? (`Yes`/`No`) |
| `ZipCodes` | string | Lista de CEPs cobertos (quando a cobertura é por ZIP) |
| `PartialCountyJustification` | string | Justificativa regulatória para cobertura parcial |

**Relações:** `ServiceAreaId` → `PlanAttributes.ServiceAreaId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Insights dos notebooks Kaggle:**
- Seguradoras com `CoverEntireState = False` e poucos condados cobertos tendem a ter redes menores e preços mais baixos na tabela `Rate`.
- Condados cobertos por apenas 1 seguradora são candidatos a efeito de monopólio — preços artificialmente inflados em comparação com condados competitivos.
- A granularidade de `ZipCodes` permite cruzar com dados populacionais do Census para identificar "desertos de cobertura" em regiões rurais.

**Relevância para o projeto:** Essencial para as **Questões 4 e 5** — determina a amplitude geográfica das redes e identifica regiões com monopólio ou baixa competição.

In [18]:
show_family(["service_area"])

### `service_area`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42103.0,No,<NA>,<NA>,42,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
1,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42105.0,No,<NA>,<NA>,43,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
2,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42107.0,No,<NA>,<NA>,44,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
3,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42109.0,No,<NA>,<NA>,45,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
4,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42111.0,No,<NA>,<NA>,46,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
5,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42113.0,No,<NA>,<NA>,47,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
6,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42115.0,No,<NA>,<NA>,48,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
7,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42117.0,No,<NA>,<NA>,49,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
8,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42119.0,No,<NA>,<NA>,50,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
9,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42127.0,No,<NA>,<NA>,51,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00


   ↳ 20 colunas | 10 linhas exibidas



In [19]:
profile_family(["service_area"])

## Perfil: `service_area`

Total de linhas na tabela: 42,247
Amostra carregada: 10,653 linhas × 20 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.16 std=0.75,cast → double/decimal
statecode,string,0 (0.0%),—,39 [média],MI(1546) | VA(1201) | PA(997) | TX(837) | GA(618),—
issuerid,numeric,0 (0.0%),—,629 [alta],min=10046.00 max=99969.00 mean=50662.26 std=26555.37,cast → double/decimal
sourcename,string,0 (0.0%),—,3 [baixa],SERFF(5109) | HIOS(5020) | OPM(524),candidata a enum/categórica
versionnum,numeric,0 (0.0%),—,23 [média],min=1.00 max=24.00 mean=7.12 std=4.38,cast → double/decimal
importdate,date,0 (0.0%),—,230 [alta],de 2013-06-04 até 2015-11-21,cast → date/timestamp
issuerid2,numeric,0 (0.0%),—,629 [alta],min=10046.00 max=99969.00 mean=50662.26 std=26555.37,cast → double/decimal
statecode2,string,0 (0.0%),—,39 [média],MI(1546) | VA(1201) | PA(997) | TX(837) | GA(618),—
serviceareaid,string,0 (0.0%),—,381 [alta],MIS001(1040) | VAS001(743) | TXS001(523) | PAS001(481) | KSS001(424),—


### ⚠️ Pontos de Atenção

  • `zipcodes`: 99% nulos — verificar se é campo opcional
  • `partialcountyjustification`: 99% nulos — verificar se é campo opcional
  • `rownumber`: outliers detectados (max=435.00, mean=59.36)



---
## 7. Tabela: BusinessRules — Regras de Negócio dos Planos

**O que é:** Define as **regras operacionais e de elegibilidade** de cada plano: idades mínima/máxima do segurado, períodos de carência, regras de pagamento e penalidades. Complementa `PlanAttributes` com informações mais operacionais — é a tabela menos explorada nas análises públicas do Kaggle.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `PlanId` | string | ID do plano |
| `EnrolleeMinimumAge` | integer | Idade mínima do segurado |
| `EnrolleeMaximumAge` | integer | Idade máxima do segurado |
| `PlanEffectiveDate` | date | Data de início de vigência do plano |
| `PlanExpirationDate` | date | Data de expiração do plano |
| `GracePeriodDetail` | string | Detalhes do período de carência para pagamento |
| `MarketCoverage` | string | Tipo de mercado: `Individual` / `Small Group` |
| `DentalOnlyPlan` | string | Plano exclusivamente odontológico (`Yes`/`No`) |
| `SubjectToMedicalManagement` | string | Sujeito a gerenciamento médico prévio |
| `FirstTierUtilization` | decimal | % de utilização de serviços de primeiro nível |
| `SecondTierUtilization` | decimal | % de utilização de serviços de segundo nível |
| `InpatientCopaymentMaximumDays` | integer | Máximo de dias com copagamento em internação |
| `BeginPrimaryCareCostSharingAfterNumberOfVisits` | integer | Visitas gratuitas antes de ativar o cost-sharing |
| `BeginPrimaryCareDeductibleCoinsuranceAfterNumberOfCopays` | integer | Copays antes de ativar o deductible/coinsurance |

**Relações:** `PlanId` → `PlanAttributes.PlanId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Relevância para o projeto:** Útil para filtrar planos elegíveis para os perfis de beneficiário analisados nas questões e para entender como `BeginPrimaryCareCostSharingAfterNumberOfVisits` impacta o custo total em tratamentos recorrentes (Questão 1 — infusão crônica).

In [20]:
show_family(["business_rules"])

### `business_rules`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0010006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
1,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
2,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020002,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",16,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
3,2014,AZ,18156,HIOS,1,2013-06-06 10:50:48,18156,35-0472300,<NA>,<NA>,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,30,1,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Grandson or Granddaughter,Yes;Adopted Child,No;Foster Child,Yes;Stepson or Stepdaughte...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
4,2014,AZ,23307,HIOS,8,2014-01-16 07:24:04,23307,61-1013183,<NA>,<NA>,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,No,Age on effective date,6,"Spouse,No;Adopted Child,No;Stepson or Stepdaughter,No;Self,No;Child,No;Dependent on a Minor Depe...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
5,2014,AZ,30045,HIOS,5,2014-03-19 07:06:49,30045,86-0274899,<NA>,<NA>,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,No,No,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Foster Child,No;Brother or Sister,No;Ward,No;Stepson or Stepdaughter...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
6,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020004,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",17,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
7,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster

   ↳ 25 colunas | 10 linhas exibidas



In [21]:
profile_family(["business_rules"])

## Perfil: `business_rules`

Total de linhas na tabela: 21,085
Amostra carregada: 5,281 linhas × 25 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.34 std=0.64,cast → double/decimal
statecode,string,0 (0.0%),—,39 [média],WI(419) | TX(409) | OH(336) | FL(322) | MI(281),—
issuerid,numeric,0 (0.0%),—,806 [alta],min=10046.00 max=99969.00 mean=53045.59 std=26016.41,cast → double/decimal
sourcename,string,0 (0.0%),—,3 [baixa],HIOS(3144) | SERFF(2050) | OPM(87),candidata a enum/categórica
versionnum,numeric,0 (0.0%),—,23 [média],min=1.00 max=24.00 mean=6.80 std=4.13,cast → double/decimal | possíveis outliers (>4σ)
importdate,date,0 (0.0%),—,230 [alta],de 2013-06-06 até 2015-11-21,cast → date/timestamp
issuerid2,numeric,0 (0.0%),—,806 [alta],min=10046.00 max=99969.00 mean=53045.59 std=26016.41,cast → double/decimal
tin,string,0 (0.0%),—,325 [alta],36-1236610(433) | 95-6042390(278) | 47-0397286(144) | 75-1233841(130) | 13-5123390(126),—
productid,string,408 (7.7%),—,"1,826 [alta]",33602TX047(71) | 33602TX046(69) | 84670WI125(32) | 88380VA073(30) | 84670WI133(29),—


### ⚠️ Pontos de Atenção

  • `versionnum`: outliers detectados (max=24.00, mean=6.80)
  • `minimumtobaccofreemonthsrule`: outliers detectados (max=12.00, mean=6.00)
  • `rownumber`: outliers detectados (max=44.00, mean=11.05)



---
## 8. Tabela: Crosswalk — Mapeamento de Planos entre Anos

**O que é:** Mapeia a **linhagem temporal dos planos** (2014 → 2015 → 2016). Como seguradoras podem renomear, reformular ou descontinuar planos a cada ciclo anual, o Crosswalk é o único arquivo que permite rastrear a evolução de um plano específico ao longo do tempo — peça central da arquitetura deste projeto.

No dataset existem dois arquivos: `Crosswalk2015.csv` (liga 2014 → 2015) e `Crosswalk2016.csv` (liga 2015 → 2016).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano de destino (ex: 2015 no Crosswalk2015) |
| `StateCode` | string | Estado |
| `IssuerId` | string | ID da seguradora no ano de destino |
| `VersionNum` | integer | Versão da submissão |
| `CrosswalkYear` | integer | Ano de origem (ex: 2014 no Crosswalk2015) |
| `CrosswalkIssuerId` | string | IssuerId do plano no ano de origem |
| `CrosswalkLevel` | string | Nível de equivalência: `Substitute`, `Renewal`, `Transition`, `New Product` |
| `CrosswalkPlanId` | string | PlanId do plano no ano de origem |
| `PlanId` | string | PlanId do plano equivalente no ano de destino |
| `CrosswalkReason` | string | Motivo da mudança (ex: mudança de rede, renomeação, fusão) |

**Como usar — rastrear um plano de 2014 ao equivalente em 2016:**
```sql
-- Passo 1: 2014 → 2015
SELECT crosswalk_plan_id AS plan_2014, plan_id AS plan_2015
FROM crosswalk_2015
WHERE crosswalk_plan_id = '12345TX1234567';

-- Passo 2: 2015 → 2016
SELECT crosswalk_plan_id AS plan_2015, plan_id AS plan_2016
FROM crosswalk_2016
WHERE crosswalk_plan_id = '<plan_2015 do passo anterior>';
```

**Relações:** `CrosswalkPlanId` → `PlanAttributes.PlanId` do ano de origem; `PlanId` → `PlanAttributes.PlanId` do ano de destino.

**Insights críticos do Kaggle:**
- ~70% dos planos de 2014 têm correspondente no Crosswalk2015 — os restantes foram descontinuados.
- Muitos planos mudam de `PlanId` entre anos **sem mudança real de produto** (apenas renomeação). Sem o Crosswalk, comparar preços ano a ano é comparar produtos diferentes.
- Planos descontinuados não aparecem como destino no Crosswalk — apenas como origem.

**Relevância para o projeto:** **Peça central da arquitetura** — sem o Crosswalk, a análise temporal de inflação médica (Questão 1) e evolução de prêmios compararia "maçãs com laranjas". O ETL de Crosswalk foi identificado como o teste unitário obrigatório do projeto (validar 100 planos que mudaram de ID entre 2015 e 2016).

In [22]:
show_family(["crosswalk"])

### `crosswalk2015`

,state,dentalplan,planid_2014,issuerid_2014,multistateplan_2014,metallevel_2014,childadultonly_2014,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,ageoffplanid_2015,issuerid_ageoff2015,multistateplan_ageoff2015,metallevel_ageoff2015,childadultonly_ageoff2015,landinzone_path,ingestion_datetime
0,AK,Y,21989AK0010001,21989,N,Low,0,2013,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
1,AK,Y,21989AK0010001,21989,N,Low,0,2016,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
2,AK,Y,21989AK0010001,21989,N,Low,0,2020,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
3,AK,Y,21989AK0010001,21989,N,Low,0,2050,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
4,AK,Y,21989AK0010001,21989,N,Low,0,2060,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
5,AK,Y,21989AK0010001,21989,N,Low,0,2068,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
6,AK,Y,21989AK0010001,21989,N,Low,0,2070,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
7,AK,Y,21989AK0010001,21989,N,Low,0,2090,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
8,AK,Y,21989AK0010001,21989,N,Low,0,2100,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
9,AK,Y,21989AK0010001,21989,N,Low,0,2105,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00


   ↳ 23 colunas | 10 linhas exibidas



### `crosswalk2016`

,state,dentalplan,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2016,issuerid_2016,multistateplan_2016,metallevel_2016,childadultonly_2016,ageoffplanid_2016,issuerid_ageoff2016,multistateplan_ageoff2016,metallevel_ageoff2016,childadultonly_ageoff2016,landinzone_path,ingestion_datetime
0,OR,N,10091OR0360004,10091,N,Bronze,0,41021,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
1,OR,N,10091OR0360004,10091,N,Bronze,0,41037,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
2,OR,N,10091OR0360004,10091,N,Bronze,0,41009,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
3,OR,N,10091OR0360004,10091,N,Bronze,0,41023,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
4,OR,N,10091OR0360004,10091,N,Bronze,0,41059,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
5,OR,N,10091OR0360004,10091,N,Bronze,0,41045,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
6,OR,N,10091OR0360004,10091,N,Bronze,0,41069,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
7,OR,N,10091OR0360004,10091,N,Bronze,0,41025,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
8,OR,N,10091OR0360004,10091,N,Bronze,0,41063,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
9,OR,N,10091OR0360004,10091,N,Bronze,0,41071,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00


   ↳ 23 colunas | 10 linhas exibidas



In [23]:
profile_family(["crosswalk"])

## Perfil: `crosswalk2015`

Total de linhas na tabela: 132,505
Amostra carregada: 33,066 linhas × 23 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
state,string,0 (0.0%),—,35 [média],TX(4245) | VA(1970) | GA(1941) | MI(1783) | FL(1641),—
dentalplan,string,0 (0.0%),—,2 [baixa],N(20297) | Y(12769),candidata a enum/categórica
planid_2014,string,0 (0.0%),—,"2,972 [alta]",47665TX0020002(78) | 55409TX0030002(74) | 33602TX0500005(74) | 33602TX0500003(73) | 23749TX00100...,—
issuerid_2014,numeric,0 (0.0%),—,299 [alta],min=10191.00 max=99969.00 mean=49314.01 std=26380.48,cast → double/decimal
multistateplan_2014,string,0 (0.0%),—,2 [baixa],N(31574) | Y(1492),candidata a enum/categórica
metallevel_2014,string,0 (0.0%),—,7 [baixa],Low(6865) | Silver(6624) | High(5904) | Bronze(5864) | Gold(5301),candidata a enum/categórica
childadultonly_2014,numeric,0 (0.0%),—,3 [baixa],min=0.00 max=2.00 mean=0.19 std=0.44,cast → double/decimal | possíveis outliers (>4σ)
fipscode,numeric,0 (0.0%),—,"2,545 [alta]",min=1001.00 max=56045.00 mean=32631.24 std=15193.69,cast → double/decimal
zipcode,numeric,0 (0.0%),—,388 [alta],min=0.00 max=88434.00 mean=1559.88 std=9906.83,cast → double/decimal | possíveis outliers (>4σ)


### ⚠️ Pontos de Atenção

  • `childadultonly_2014`: outliers detectados (max=2.00, mean=0.19)
  • `zipcode`: outliers detectados (max=88434.00, mean=1559.88)
  • `childadultonly_2015`: outliers detectados (max=2.00, mean=0.10)
  • `issuerid_ageoff2015`: outliers detectados (max=99969.00, mean=7257.19)



## Perfil: `crosswalk2016`

Total de linhas na tabela: 150,005
Amostra carregada: 37,686 linhas × 23 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
state,string,0 (0.0%),—,37 [média],TX(4315) | GA(2729) | TN(2089) | OH(1916) | MI(1697),—
dentalplan,string,0 (0.0%),—,2 [baixa],N(26366) | Y(11320),candidata a enum/categórica
planid_2015,string,0 (0.0%),—,"4,398 [alta]",19312TX0010003(78) | 33602TX0500002(77) | 61315TX0010001(77) | 24349TX0010006(76) | 23749TX00100...,—
issuerid_2015,numeric,0 (0.0%),—,406 [alta],min=10091.00 max=99969.00 mean=50396.12 std=26963.86,cast → double/decimal
multistateplan_2015,string,0 (0.0%),—,2 [baixa],N(35938) | Y(1748),candidata a enum/categórica
metallevel_2015,string,0 (0.0%),—,7 [baixa],Silver(9805) | Bronze(7704) | Low(6614) | Gold(6152) | High(4706),candidata a enum/categórica
childadultonly_2015,numeric,0 (0.0%),—,3 [baixa],min=0.00 max=2.00 mean=0.10 std=0.32,cast → double/decimal | possíveis outliers (>4σ)
fipscode,numeric,0 (0.0%),—,"2,597 [alta]",min=1001.00 max=56045.00 mean=32860.24 std=14973.12,cast → double/decimal
zipcode,numeric,0 (0.0%),—,30 [média],min=0.00 max=49285.00 mean=106.75 std=2210.77,cast → double/decimal | possíveis outliers (>4σ)


### ⚠️ Pontos de Atenção

  • `childadultonly_2015`: outliers detectados (max=2.00, mean=0.10)
  • `zipcode`: outliers detectados (max=49285.00, mean=106.75)
  • `childadultonly_2016`: outliers detectados (max=2.00, mean=0.07)
  • `issuerid_ageoff2016`: outliers detectados (max=99969.00, mean=3659.25)



---
## 9. Tabela: Network — Rede de Prestadores

**O que é:** Relaciona cada seguradora à(s) sua(s) **rede(s) de prestadores** (hospitais, clínicas, médicos). É a menor tabela do dataset — contém apenas os metadados da rede, não a lista individual de prestadores. Múltiplos planos de uma mesma seguradora podem compartilhar o mesmo `NetworkId`.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `SourceName` | string | Fonte/exchange |
| `VersionNum` | integer | Versão da submissão |
| `ImportDate` | date | Data de importação |
| `IssuerId2` | string | ID alternativo |
| `FederalTIN` | string | TIN federal |
| `NetworkId` | string | Identificador único da rede — chave para `PlanAttributes.NetworkId` |
| `NetworkName` | string | Nome comercial da rede (ex: "Blue Shield Trio HMO Network") |
| `NetworkURL` | string | URL do diretório público de prestadores in-network |

**Relações:** `NetworkId` → `PlanAttributes.NetworkId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Limitação importante:** O dataset do Kaggle **não inclui a lista individual de prestadores** — apenas os metadados da rede. A `NetworkURL` aponta para portais externos com os diretórios de prestadores, mas estão fora do escopo do dataset. Para este projeto, o `NetworkId` serve como **proxy da amplitude da rede**: seguradoras com múltiplos `NetworkId` distintos ou nomes de rede associados a áreas geográficas menores tendem a ter redes mais restritas.

**Relevância para o projeto:** Diretamente relevante para a **Questão 4** — a relação entre amplitude da rede de prestadores e o preço final do plano.

In [24]:
show_family(["network"])

### `network`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,ODS Premier,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
1,2014,AK,38344,HIOS,6,2013-08-28 08:15:53,38344,AK,HeritagePlus,AKN001,https://www.premera.com/wa/visitor/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
2,2014,AK,38536,HIOS,2,2013-08-01 12:48:00,38536,AK,Lincoln Dental Connect,AKN001,http://lfg.go2dental.com/member/dental_search/searchprov.cgi?P=LFGDentalConnect&Network=L,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
3,2014,AK,84859,HIOS,1,2013-06-06 10:50:48,84859,AK,Ameritas PPO Dental Network,AKN001,www.standard.com,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
4,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,DenteMax,ALN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
5,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,CONNECTION Dental through PPO USA,ALN002,http://www.ppousa.com/patients/index.html,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
6,2014,AK,42507,HIOS,3,2013-09-02 11:39:25,42507,AK,DentalGuard Preferred,AKN001,https://www.guardiananytime.com/fpapp/FPWeb/dentalSearch.process,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
7,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus AK Regional,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
8,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus Providence,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
9,2014,AK,74819,HIOS,7,2014-01-21 08:29:49,74819,AK,DenteMax,AKN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00


   ↳ 16 colunas | 10 linhas exibidas



In [25]:
profile_family(["network"])

## Perfil: `network`

Total de linhas na tabela: 3,822
Amostra carregada: 931 linhas × 16 colunas
Linhas duplicadas na amostra: 0 (0.0%) ✅



,tipo_inferido,nulos,sentinelas,únicos,stats / top valores,tratamentos sugeridos
coluna,,,,,,
businessyear,numeric,0 (0.0%),—,3 [baixa],min=2014.00 max=2016.00 mean=2015.15 std=0.79,cast → double/decimal
statecode,string,0 (0.0%),—,39 [média],IL(55) | OH(52) | MI(51) | PA(49) | VA(48),—
issuerid,numeric,0 (0.0%),—,516 [alta],min=10091.00 max=99969.00 mean=54669.30 std=26773.57,cast → double/decimal
sourcename,string,0 (0.0%),—,3 [baixa],HIOS(458) | SERFF(443) | OPM(30),candidata a enum/categórica
versionnum,numeric,0 (0.0%),—,21 [média],min=1.00 max=24.00 mean=5.88 std=3.61,cast → double/decimal | possíveis outliers (>4σ)
importdate,date,0 (0.0%),—,187 [alta],de 2013-06-05 até 2015-11-19,cast → date/timestamp
issuerid2,numeric,0 (0.0%),—,516 [alta],min=10091.00 max=99969.00 mean=54669.30 std=26773.57,cast → double/decimal
statecode2,string,0 (0.0%),—,39 [média],IL(55) | OH(52) | MI(51) | PA(49) | VA(48),—
networkname,string,0 (0.0%),—,448 [alta],Ameritas PPO Dental Network(31) | DenteMax(30) | Dentegra Dental PPO(28) | Renaissance Dental(27...,—


### ⚠️ Pontos de Atenção

  • `versionnum`: outliers detectados (max=24.00, mean=5.88)
  • `rownumber`: outliers detectados (max=35.00, mean=13.77)



---
## 10. Tabelas Adicionais

Tabelas presentes no database que não se encaixam nas famílias principais acima. Podem incluir arquivos auxiliares, variantes ou tabelas de controle criadas durante a ingestão.

In [26]:
# Identifica tabelas que não foram cobertas pelas seções anteriores
known_patterns = ["rate", "plan_attributes", "benefits_cost_sharing", "benefits",
                  "service_area", "business_rules", "crosswalk", "network"]

other_tables = [
    t for t in all_table_names
    if not any(p in t for p in known_patterns)
]

if other_tables:
    print(f"Tabelas adicionais encontradas: {other_tables}")
    show_family(other_tables)
else:
    print("Não há tabelas adicionais fora das famílias mapeadas.")

Não há tabelas adicionais fora das famílias mapeadas.


---
## 11. Resumo das Relações entre Tabelas

```
Crosswalk2015 / Crosswalk2016
  CrosswalkPlanId ──► PlanId (no ano seguinte)
                            │
                            ▼
ServiceArea ──────► PlanAttributes ◄──── BenefitsCostSharing
(ServiceAreaId)      │  (PlanId)               (PlanId)
                     │
Network ─────────────┤ (NetworkId)
                     │
                     ├──── PlanId ──── Rate
                     │                  └── RatingAreaId → ServiceArea
                     │
                     └──── PlanId ──── BusinessRules
```

**Chave central:** `PlanId` no formato HIOS — `IssuerId(5) + ProductId(2) + PlanId(2) + VariantId(2)`. Liga todas as tabelas analíticas.

### Mapeamento para as Questões do Projeto

| Questão | Tabelas Primárias | Tabelas de Apoio |
|---|---|---|
| Q1 — Custos oncológicos (Copay × Coinsurance 2014–2016) | `BenefitsCostSharing` | `PlanAttributes` (MetalLevel), `Crosswalk` |
| Q2 — Competição × Preço médio por estado | `Rate`, `PlanAttributes` | `ServiceArea` |
| Q3 — Benefícios como variável de precificação | `BenefitsCostSharing`, `Rate` | `PlanAttributes` |
| Q4 — Tamanho da rede × Preço | `Network`, `Rate` | `PlanAttributes`, `ServiceArea` |
| Q5 — Monopólios e desigualdade geográfica | `ServiceArea`, `Rate` | `PlanAttributes` |
| Rastreio temporal (linhagem) | `Crosswalk2015`, `Crosswalk2016` | `PlanAttributes`, `Rate` |

---
## 12. Qualidade dos Dados — Pontos de Atenção para o ETL Silver

Com base nas análises públicas do Kaggle e na documentação do CMS, estes são os principais problemas de qualidade conhecidos que o ETL da camada Silver deve tratar:

| # | Tabela | Problema | Impacto | Tratamento sugerido |
|---|---|---|---|---|
| 1 | `BenefitsCostSharing` | Campos `CopayInnTier1/2` e `CoinsInnTier1/2` chegam como strings mistas: `"$30"`, `"$0"`, `"No Charge"`, `"Not Applicable"` | Impossível agregar sem normalização | Parsear para decimal; mapear `"No Charge"` → `0.0`, `"Not Applicable"` → `null` |
| 2 | `BenefitsCostSharing` | `CopayInnTier2` e `CoinsInnTier2` têm alta taxa de nulos — muitos planos operam apenas com Tier 1 | Nulos não são ausência de dado, são "não aplicável" | Distinguir `null` legítimo de dado faltante; não imputar |
| 3 | `PlanAttributes` | Variantes CSR do mesmo plano (sufixo `73`, `87`, etc. no `PlanId`) representam o mesmo produto com deductibles reduzidos | Joins podem duplicar linhas de preço | Agrupar variantes pelo `HIOSProductId` antes de joins com `Rate` |
| 4 | `Rate` | ~50M linhas no total — sem particionamento adequado, scans completos são proibitivos no Athena | Custo e latência altos | Particionamento por `StateCode` + `BusinessYear` (já aplicado na Bronze via Iceberg) |
| 5 | `Crosswalk` | Planos descontinuados aparecem como `CrosswalkPlanId` sem `PlanId` de destino | Left joins retornam nulos para planos extintos | Tratar nulos como descontinuações; não descartar esses registros |
| 6 | `Rate` | Valores extremos em `IndividualRate`: registros com `0.0` ou valores acima de `$5.000/mês` | Distorce médias sem filtro | Aplicar filtro de range razoável na Silver (ex: `> 0` e `< 3000`) |
| 7 | Geral | Colunas `VersionNum` e `ImportDate` — quando há múltiplas versões do mesmo arquivo, a versão mais recente substitui a anterior | Dados duplicados com versões diferentes | Deduplica pelo `PlanId` + ano mantendo o maior `VersionNum` |